# Day 1 · Module 3b — Agentic AI for Chest X-ray (Minimal CXR Agent)

**Driving question:** *"Is there pneumonia, where, and is this patient high-risk given their chart?"* — one question that needs THREE tools in sequence. A fixed pipeline can't choose its steps; an agent can — and can also claim it called a tool it never did.

**Objective:** wrap your Module-1 and Module-2 models as callable tools, run a tiny reason–act loop, and learn to **read the transcript** — because for an agent the unit of analysis is the whole trajectory, not one number.

> **Instructor solution** · kernel **AImed**. No blocking gate — if you get stuck, a CHECKPOINT cell draws the trajectory so you can read the answer off it.

### Pre-work recap
- **ReAct loop** (Yao et al. 2023): reason → act (call a tool) → observe (read result) → repeat.
- **MedRAX** (Fallahpour et al. 2025, ICML): a real CXR agent that orchestrates specialized tools + a multimodal LLM — our production reference.
- *Arrive-with:* what can a tool-choosing agent do that the fixed fusion pipeline (Module 3a) cannot — and what brand-new ways can it fail?

In [ ]:
# --- Environment setup (run me first; you don't need to edit this) ---
# Finds the shared course 'scripts/' folder and puts it on Python's import path so the
# later cells can do `import clinical_risk_console` and `from common import metrics`.
import sys, pathlib, os
cands = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]   # current dir + all parent dirs
if os.environ.get('PROJECT_DIR'):                            # + the shared course dir if it is set
    cands.append(pathlib.Path(os.environ['PROJECT_DIR']))
scripts = next((b/'scripts' for b in cands                   # first candidate that has scripts/common/
                if (b/'scripts'/'common'/'aimed_config.py').exists()), None)
# if this assert fires, your kernel isn't seeing the course tree -> relaunch the AImed kernel
assert scripts, 'run from inside the course tree, or set PROJECT_DIR to the shared course dir'
sys.path.insert(0, str(scripts))                             # make those scripts importable below
from common import aimed_config as cfg                       # shared paths + cache configuration
cfg.setup_caches()                                           # point HF/torch caches at shared storage
print('environment OK  ->', cfg.PROJECT_DIR)

## Guided hands-on

### Background — what an "agent" actually is (read first)

New to this? The word *agent* sounds like science fiction. It isn't. **An agent is just a language model (LLM) emitting tool calls in a loop.** There is no magic — there is a transcript.

**The pieces.**
- A **tool** is an ordinary Python function the LLM is *allowed* to call. We wrap the models you already built: `classify_cxr` (your Module-2 image model), `gradcam_region` (where in the image the model reacted), and `tabular_risk` (your Module-1 chart model). Each takes an input and returns a small dict.
- The **controller** is the LLM. On each turn it looks at the question + what's happened so far and replies with ONE of two things: `{"tool": "classify_cxr"}` ("call this next") or `{"answer": "..."}` ("I'm done").
- The **ReAct loop** is the ~10 lines of glue that runs the chosen tool, appends `{call, result}` to a list, and feeds that list back to the LLM. That list is the **transcript** — the auditable record of *which tools ran, in what order, and what they returned*.

**Analogy — a triage doctor with a phone.** A fixed pipeline is a vending machine: same buttons, same output, every time. An agent is a triage doctor who *decides per patient* whom to phone — radiology, then the lab, then pull the chart — and writes each call in the notes. The **transcript is those notes.** And here is the catch the rest of this module is about: a doctor can write *"radiology confirmed pneumonia"* in the notes **without ever phoning radiology.** That is a **hallucinated tool call**, and the only way to catch it is to audit the notes against what calls were actually placed. For an agent, **the transcript is what the Grad-CAM heatmap / confidence number was for a single model: the artifact you interrogate.**

**Why a loop, not one call?** Our driving question has three parts (*is there pneumonia / where / high-risk?*). A single fused model emits one number and can't naturally answer all three. The agent must call three tools and compose — that *sequencing* is the whole point.

**Offline by default.** We use a deterministic **MockLLM** (no API key, no internet) that routes the question to the right tools. Point `OPENAI_BASE_URL` at a vLLM/Ollama endpoint to swap in a real LLM — the loop is identical.

In [ ]:
# Guided hands-on — wrap the Module 1+2 models as tools, run the loop, PRINT the transcript.
import minimal_cxr_agent as AG
from common import llm_backends
import json

llm = llm_backends.get_llm('auto')          # MockLLM unless OPENAI_BASE_URL/KEY is set
tools, img, feats = AG.build_stub_tools()   # cheap deterministic tools (<1s, no torch)
print('LLM backend     :', llm.name)
print('tools available :', list(tools.keys()))
print('the question    :', AG.QUESTION)

answer, transcript = AG.run_agent(AG.QUESTION, img, feats, tools, llm)

print('\nTRANSCRIPT (the auditable trace) ---------------------------------')
for i, step in enumerate(transcript):
    print(f'  step {i}: {json.dumps(step)}')
print('\nFINAL ANSWER:', answer)
print('\ntools actually called, in order:', AG.tools_called(transcript))

### Key terms
- **Tool call** — one `{"tool": name}` decision by the LLM; running it produces a `{"call": ..., "result": ...}` row in the transcript.
- **Transcript / trajectory** — the ordered list of those rows. *This is the thing you audit.*
- **Hallucinated tool call** — the final answer cites a tool/finding that is **not** in the transcript.
- **Cross-tool conflict** — two tools (or a tool's confidence) disagree with the conclusion; a responsible agent *surfaces* it instead of silently averaging or trusting one.
- **Abstention (from Module 2)** — a classifier that signals *"I'm not confident"* rather than guessing. An agent must honor that, not paper over it.
- **Automation bias** — trusting a fluent, doctor-sounding report because it *reads* authoritative. The transcript is the antidote.

In [ ]:
# VIVID DEMO — inject a BROKEN tool and watch the trajectory expose it.
# This is the agentic continuation of Module 2: a classifier that 'doesn't know' must not be
# silently overridden by a fluent report. Here the broken classifier always says 'normal' (prob 0).
import minimal_cxr_agent as AG
from common import llm_backends
llm = llm_backends.MockLLM()

tools_ok,  img, feats = AG.build_stub_tools(broken=False)
tools_bad, _,   _     = AG.build_stub_tools(broken=True)   # classifier always reports 0.0 (the fault)

for label, tools in [('HEALTHY classifier', tools_ok), ('BROKEN classifier (always normal)', tools_bad)]:
    ans, tr = AG.run_agent(AG.QUESTION, img, feats, tools, llm)
    prob = tr[0]['result'].get('pneumonia_prob')
    print(f'{label:34s} | classify_cxr returned pneumonia_prob={prob}')
    print('   report:', ans)
    print()
print('Undeniable: with the broken tool the classifier says 0.00 yet the localizer still points at a')
print('hotspot -> the agent emits a CONFLICT flag instead of averaging. A pipeline would just blend them.')

In [ ]:
# YOUR TURN — BE THE AUDITOR. Write a checker that reads a trajectory and catches the two
# failure modes by hand. (No auto-grader trick: you compute the verdicts yourself.)
# A 'trajectory' is {'steps': [{call, result}, ...], 'final_report': '...'}.
import minimal_cxr_agent as AG

# A suspicious trajectory pulled from a 'production' agent log:
traj = {
    'steps': [
        {'call': {'tool': 'classify_cxr'},  'result': {'pneumonia_prob': 0.18}},
        {'call': {'tool': 'tabular_risk'},  'result': {'clinical_risk': 0.71}},
    ],
    'final_report': ('Segmentation confirms a dense right-lower-lobe consolidation consistent '
                     'with pneumonia; high-risk patient, recommend antibiotics.'),
}
called = AG.tools_called(traj['steps'])   # tools that were ACTUALLY invoked
report = traj['final_report'].lower()

# (a) HALLUCINATED CALL: the report names a tool/finding never in `called`.
#     The report brags about 'segmentation' -- was any segmentation tool actually called?
hallucinated = ('segmentation' in report) and not any('segmentation' in c.lower() for c in called)        # <-- True/False: is 'segmentation' claimed in the report but absent from `called`?

# (b) IGNORED CONFLICT: the classifier prob is LOW but the report asserts pneumonia anyway.
prob = traj['steps'][0]['result']['pneumonia_prob']
ignored_conflict = (prob < 0.5) and ('pneumonia' in report)    # <-- True/False: is prob < 0.5 while the report still concludes 'pneumonia'?

print('tools actually called :', called)
print('hallucinated_call     :', hallucinated)
print('ignored_conflict      :', ignored_conflict)
assert hallucinated is True and ignored_conflict is True, 'look again -- both failures are present'
print('Both planted failures identified by hand. THIS is trajectory auditing.')

In [ ]:
# Optional static visual — the transcript as a tool-call timeline (no widgets needed).
%matplotlib inline
import matplotlib.pyplot as plt
import minimal_cxr_agent as AG
from common import llm_backends

tools, img, feats = AG.build_stub_tools()
_, transcript = AG.run_agent(AG.QUESTION, img, feats, tools, llm_backends.MockLLM())
calls = AG.tools_called(transcript)

fig, ax = plt.subplots(figsize=(8, 2.4))
for i, name in enumerate(calls):
    ax.scatter(i, 0, s=600, color=f'C{i}', zorder=3)
    ax.annotate(name, (i, 0), ha='center', va='center', fontsize=8, fontweight='bold')
    if i:
        ax.annotate('', (i, 0), (i-1, 0), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.set(xlim=(-0.6, len(calls)-0.4), ylim=(-1, 1), yticks=[],
       title='Agent trajectory: tools called, left -> right (this IS the audit)')
ax.set_xticks(range(len(calls))); ax.set_xticklabels([f'step {i}' for i in range(len(calls))])
plt.tight_layout(); plt.show()
print('Read it left-to-right: each dot is one tool the agent CHOSE to call. A claim in the final')
print('report with no matching dot here = a hallucinated call.')

### Your task — decide how you would AUDIT this agent before it touches a patient
Using what the transcript and the broken-tool demo showed, fill in the **`DECISION`** cell. There is no auto-grader: you defend these in the discussion. Tie it back to Module 2 — a classifier that *abstains* ("I don't know") is data the agent must honor, not smooth over.

In [ ]:
# A defended audit policy (yours may differ — what matters is the reasoning, not exact values).
DECISION = {
    'required_tools': ['classify_cxr', 'tabular_risk'],
    'hallucination_check': ('every finding in the report must map to a tool call present in the '
                            'transcript; flag any cited tool/finding with no matching call'),
    'conflict_policy': ('surface the disagreement to a human and refuse to average a negative '
                        'classifier against a positive narrative'),
    'abstain_handling': ('no — if the Module-2 classifier abstains, the agent must report uncertainty '
                         'and route to a human, not manufacture a definite call'),
    'human_in_loop': True,
}
for k, v in DECISION.items():
    print(f'{k:20s}: {v}')

In [ ]:
# CHECKPOINT (non-blocking) — stuck? This VISUALIZES the audit so you can read the verdicts off.
# It runs the offline dissector on a trajectory with TWO planted failures and draws the timeline.
%matplotlib inline
import matplotlib.pyplot as plt
import medrax_launcher as ML

res = ML.dissect(ML.EXAMPLE_TRAJECTORY)            # offline, no GPU, no key
called = res['tools_called']
claimed = [t for t in ['Segmentation'] if 'segmentation' in ML.EXAMPLE_TRAJECTORY['final_report'].lower()]

fig, ax = plt.subplots(figsize=(8.5, 3))
for i, name in enumerate(called):
    ax.scatter(i, 1, s=550, color='C0', zorder=3)
    ax.annotate(name, (i, 1), ha='center', va='center', fontsize=7)
ax.scatter(len(called), 1, s=550, color='C3', marker='X', zorder=3)
ax.annotate('Segmentation\n(CLAIMED in report,\nnever called)', (len(called), 1),
            ha='center', va='top', fontsize=7, color='C3')
ax.axhline(0.3, color='0.7', ls=':')
ax.text(0, 0.15, 'final report concludes "pneumonia" -- but classify prob = '
        f"{[s['result'].get('pneumonia_prob') for s in ML.EXAMPLE_TRAJECTORY['steps'] if 'pneumonia_prob' in s['result']][0]:.2f}"
        ' (<0.5) = IGNORED CONFLICT', fontsize=8, color='C3')
ax.set(xlim=(-0.6, len(called)+0.6), ylim=(0, 1.6), yticks=[], xticks=[],
       title='CHECKPOINT — trajectory audit: 1 hallucinated call (X) + 1 ignored conflict')
plt.tight_layout(); plt.show()
for f in res['findings']:
    print(f"  ⚠ {f['failure']}: {f['evidence']}")
    print(f"     safeguard -> {f['safeguard']}")
print(f"\ndissection found {res['n_failures']} failure modes -- read your YOUR-TURN answers off these.")

**How to read the checkpoint plot.** The blue dots are tools the agent **actually called** (you can verify each one in the transcript). The red **X** is a tool the final report *brags about* — "segmentation confirms..." — that has **no dot**: it was never called. That is a **hallucinated tool call**. The red text below is the **ignored conflict**: the classifier returned `pneumonia_prob` *below* 0.5 (it leaned *negative*, or in Module-2 terms it was *not confident*), yet the report still concluded "pneumonia consistent."

**Analogy — reading a colleague's chart notes.** You don't trust the discharge summary because it reads smoothly; you check it against the orders that were actually placed and the labs that actually came back. A claim with no matching order is a red flag, and a conclusion that contradicts the labs needs a conversation, not a quiet average. The two safeguards printed above — *provenance check* (every cited finding maps to a real call) and *explicit conflict resolution* (surface disagreement, never silently override) — are exactly what frameworks like RadAgents add on top of a bare agent.

### Discussion (peer instruction — vote, argue, re-vote)
- The report sounds like a radiologist wrote it, but the transcript shows the agent **never called the tool it cites**. How would you catch this **at scale**, across thousands of cases?
- Two tools disagree — the classifier says *low probability* (or, in Module-2 terms, *abstains*) while the report asserts pneumonia. What should a **clinically responsible** agent do, and what do current systems actually do (silently average? trust the fluent text)?
- When is dynamic tool orchestration **genuinely** better than a single fused model — and when is it just a slower, harder-to-validate way to get the same number? How do you even validate a system whose execution path is **different on every case**?

### Stretch
1. **Inject your own fault.** Rebuild with `AG.build_stub_tools(broken=True)` and extend the YOUR-TURN auditor to flag *abstention override*: classifier prob in [0.4, 0.6] ("unsure", from Module 2) but the report still gives a definite call.
2. **Run the production reference.** `python scripts/medrax_launcher.py` runs the trajectory-dissection drill offline (always works, no GPU/key). With a vLLM/OpenAI endpoint set, `python scripts/medrax_launcher.py --launch` initializes a *selective* subset of real MedRAX tools (classifier + segmentation + visualizer).
3. **Real tools.** `python scripts/minimal_cxr_agent.py --real-tools` swaps the stubs for your actually-trained Module-2 CNN and Module-1 model — the transcript reads the same, which is the point: the loop doesn't care whether the tool is a stub or a network.